# Persona-Based Anime Chatbot
A Transformer-based persona chatbot supporting multiple built-in and custom characters, running entirely free on Google Colab with persistent memory support through Google Drive.

Architecture

* `config/config.json` : model hyperparameters, memory settings, and Colab Drive memory path
* `config/characters.json` : built-in and custom character definitions
* `src/persona.py` : loads character config and builds gender-aware persona prompts
* `src/character_select.py` : interactive character and user gender selection helpers
* `src/custom_ai.py` : custom AI creation, validation, and dialogue example tools
* `src/memory.py` : rolling conversation memory with JSON save/load support
* `src/model.py` : Transformer wrapper composing persona, memory, and dialogue history
* `data/` : reference dialogue files for built-in and custom characters
* `memory/` : local fallback memory folder outside Colab


## Section 1 Setup & Installation

Auto-detects Colab, clones or updates the project if needed, removes the pre-installed `torchvision` package that can conflict with `transformers`, and installs the project dependencies.


In [ ]:
# Colab setup. This cell finds or clones the project, moves into it, then installs dependencies.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/XVentCrossX/TestingAI.git'
REPO_DIR = 'TestingAI'

REQUIRED_PROJECT_PATHS = [
    Path('src'),
    Path('config'),
    Path('data'),
    Path('config/characters.json'),
    Path('config/config.json'),
    Path('requirements.txt'),
]

def looks_like_project(path):
    path = Path(path)
    return all((path / required).exists() for required in REQUIRED_PROJECT_PATHS)

def project_candidates():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, Path('/content') / REPO_DIR, Path('/content') / 'AI-Girlfriend', Path('/content') / 'TestingAI', Path(REPO_DIR)]
    if Path('/content').exists():
        candidates.extend(path for path in Path('/content').iterdir() if path.is_dir())
    seen = set()
    for candidate in candidates:
        resolved = candidate.resolve() if candidate.exists() else candidate
        if str(resolved) not in seen:
            seen.add(str(resolved))
            yield resolved

def find_project_root():
    for candidate in project_candidates():
        if looks_like_project(candidate):
            return candidate
        nested = candidate / 'AI-Girlfriend'
        if looks_like_project(nested):
            return nested
    return None

project_root = find_project_root()
if project_root is None:
    repo_path = (Path('/content') / REPO_DIR) if Path('/content').exists() else Path(REPO_DIR)
    if repo_path.exists():
        subprocess.run(['git', '-C', str(repo_path), 'pull', '--ff-only'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(repo_path)], check=True)
    project_root = find_project_root()

if project_root is None:
    raise FileNotFoundError(
        'Could not find a complete project checkout. The cloned repo must include '
        'requirements.txt, src/, data/, config/config.json, and config/characters.json. '
        'Push/sync the latest V1.3 files to GitHub, then rerun this cell.'
    )

os.chdir(project_root)
print('Project root:', Path.cwd())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Dependencies installed.')

The next cell verifies that the notebook is running inside a complete project checkout. It checks the required source, data, and configuration files before any chatbot modules are imported.


In [ ]:
from pathlib import Path
import json
import os
import sys

required_paths = [
    Path('src'),
    Path('config'),
    Path('data'),
    Path('config/characters.json'),
    Path('config/config.json'),
    Path('requirements.txt'),
]

def looks_like_project(path):
    path = Path(path)
    return all((path / required).exists() for required in required_paths)

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, Path('/content') / 'AI-Girlfriend', Path('/content') / 'TestingAI']
    if Path('/content').exists():
        candidates.extend(path for path in Path('/content').iterdir() if path.is_dir())
    for candidate in candidates:
        if looks_like_project(candidate):
            return candidate.resolve()
        nested = candidate / 'AI-Girlfriend'
        if looks_like_project(nested):
            return nested.resolve()
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    checked = ', '.join(str(path) for path in [Path.cwd(), Path('/content') / 'TestingAI', Path('/content') / 'AI-Girlfriend'])
    raise FileNotFoundError('Could not locate a complete project root. Checked: ' + checked)

os.chdir(PROJECT_ROOT)
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required project paths: ' + ', '.join(missing))

for folder in [Path('data/custom_dialogues'), Path('memory')]:
    folder.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Project validation passed.')

Project modules are imported after validation so the notebook can safely load characters, create custom AI profiles, manage memory, and build the chatbot.


In [ ]:
from pathlib import Path
import json

from src.character_select import load_characters
from src.custom_ai import (
    append_dialogue_examples,
    count_dialogue_pairs,
    create_custom_ai,
    ensure_custom_dialogue_file,
    save_custom_character,
)
from src.memory import ConversationMemory
from src.model import PersonaChatbot
from src.persona import Persona

CONFIG_PATH = Path('config/config.json')
CHARACTERS_PATH = Path('config/characters.json')

with CONFIG_PATH.open('r', encoding='utf-8') as f:
    config = json.load(f)

print('Imports ready.')

## Section 2 Persona Configuration

Choose how the chatbot persona will be loaded. You can use one of the built-in characters, create a new custom AI, or load an existing custom AI from `config/characters.json`.

**How to configure the persona:**
1. Run the cell below.
2. Select an existing character, create a custom AI, or load a custom AI.
3. Choose the user gender option when prompted.
4. Confirm the selected character name and character id before continuing.


In [ ]:
def show_character_options(characters):
    print('Available characters:')
    for index, character in enumerate(characters, 1):
        label = 'custom' if character.get('is_custom') else character.get('dere_type', '')
        print(f"[{index}] {character['id']} - {character['display_name']} ({label})")

characters = load_characters(CHARACTERS_PATH)
print('[1] Use existing character')
print('[2] Create custom AI')
print('[3] Load custom AI')
mode = input('Choose mode [1-3]: ').strip() or '1'

if mode == '2':
    existing_ids = [character['id'] for character in characters]
    custom_character = create_custom_ai(existing_ids=existing_ids)
    character = save_custom_character(custom_character, CHARACTERS_PATH)
    dialogue_path = ensure_custom_dialogue_file(character)
    print(f"Created {character['display_name']} with dialogue file: {dialogue_path}")
elif mode == '3':
    custom_characters = [character for character in characters if character.get('is_custom')]
    if not custom_characters:
        raise ValueError('No custom characters found yet. Run mode 2 first.')
    show_character_options(custom_characters)
    selected = int(input(f'Choose custom character [1-{len(custom_characters)}]: ').strip()) - 1
    character = custom_characters[selected]
else:
    builtin_characters = [character for character in characters if not character.get('is_custom')]
    show_character_options(builtin_characters)
    selected = int(input(f'Choose built-in character [1-{len(builtin_characters)}]: ').strip() or '1') - 1
    character = builtin_characters[selected]

user_gender = input('Your gender for tone selection [male/female/neutral]: ').strip().lower() or 'neutral'
if user_gender not in {'male', 'female', 'neutral'}:
    user_gender = 'neutral'

character_id = character['id']
print('Selected:', character['display_name'])
print('Character id:', character_id)
print('User gender:', user_gender)

The persona preview shows the greeting and the prompt prefix that will be injected into each model turn. Use this preview to confirm that the character, tone, relationship mode, and memory instructions look correct before loading the model.


In [ ]:
persona = Persona.from_config(CONFIG_PATH, character_id=character_id, user_gender=user_gender)

print('Name:', persona.name)
print('Gender:', persona.character_gender)
print('Age:', persona.age)
print('Type:', persona.dere_type)
print('Relationship mode:', persona.relationship_mode)
print('Language style:', persona.language_style)
print('Response length:', persona.response_length)
print('Greeting:', persona.greeting)
print('Dialogue file:', persona.dialogue_file)
print()
print('--- System prompt preview ---')
print(persona.build_prefix())

## Section 3 Talk to Partner Bot

Before the model is loaded, the notebook prepares the memory file for the selected character. In Colab, Google Drive is mounted so the memory file survives runtime resets. Outside Colab, the notebook falls back to the local project `memory/` folder.


In [ ]:
memory_cfg = config.get('memory', {})
local_memory_dir = Path(memory_cfg.get('memory_dir', 'memory'))
drive_memory_dir = Path(memory_cfg.get('drive_memory_dir', '/content/drive/MyDrive/AI_Partner/memory'))

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    memory_dir = drive_memory_dir
    print('Using Google Drive memory folder:', memory_dir)
else:
    memory_dir = local_memory_dir
    print('Using local memory folder:', memory_dir)

memory_dir.mkdir(parents=True, exist_ok=True)
memory_path = memory_dir / f'{character_id}_memory.json'

memory = ConversationMemory.from_config(CONFIG_PATH)
if memory_path.exists():
    load_existing = input(f'Load saved memory from {memory_path}? [y/N]: ').strip().lower()
    if load_existing == 'y':
        memory.load_memory(memory_path)
        print('Loaded memory turns:', memory.turn_count)
else:
    print('No saved memory yet for this character.')

print('Memory file:', memory_path)

This is the expensive cell. It loads the selected base model and attaches the configured persona plus the saved memory object. A Colab T4 GPU is recommended for the default 7B model.


In [ ]:
chatbot = PersonaChatbot.from_config(
    CONFIG_PATH,
    persona=persona,
    memory=memory,
    character_id=character_id,
    user_gender=user_gender,
)
chatbot.save_memory_path = memory_path
print('Loaded model:', chatbot.model_name)

Optional generation settings can be adjusted without reloading the model. Leave these values unchanged for the default V1.3 behavior, or tune them before starting the chat loop.


In [ ]:
# Tune generation without reloading the model. Leave values as-is or edit them.
chatbot.update_generation_settings(
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=120,
    repetition_penalty=1.1,
)
print('Generation settings updated.')

**How to chat:**
1. Run the cell below.
2. The selected character greets you first.
3. Type your message in the input box and press Enter.
4. Each reply is saved into the character memory file automatically.
5. Type **`exit`**, **`quit`**, or **`stop`** to end the chat.


In [ ]:
print(persona.greeting)
print("Type 'exit', 'quit', or 'stop' to end the chat.")

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in {'exit', 'quit', 'stop'}:
        break
    if not user_input:
        continue
    reply = chatbot.generate(user_input, record=True)
    print(f'{persona.name}: {reply}')
    chatbot.memory.save_memory(memory_path, character_id=character_id)

print('Memory saved to:', memory_path)

## Section 4 Persistent Memory

Shows the current rolling summary and recent conversation turns for the selected character. The saved JSON contains the character id, summary, recent history, and turn count.

In Colab, the permanent memory file is stored in Google Drive at:

`/content/drive/MyDrive/AI_Partner/memory/<character_id>_memory.json`


In [ ]:
print('Summary:')
print(memory.summary or '(empty)')
print()
print('Recent turns:')
for turn in memory.get_recent():
    print(f"{turn['role']}: {turn['content']}")

# Uncomment one of these actions when needed.
# memory.save_memory(memory_path, character_id=character_id)
# memory.load_memory(memory_path)
# memory.clear_memory(memory_path)

## Section 5 Custom Dialogue Examples

Use this section for custom characters. The examples are appended to the selected custom character's dialogue file using a simple `USER:` / `CHARACTER:` format. Built-in dialogue files are treated as reference data, so avoid editing them unless you intentionally want to change those references.


In [ ]:
dialogue_path = Path(persona.dialogue_file)
if not character.get('is_custom'):
    print('Selected character is built-in. Create or load a custom AI before appending custom examples.')
else:
    print('Dialogue file:', dialogue_path)
    print('Current pair count:', count_dialogue_pairs(dialogue_path))
    examples_text = """USER: I had a difficult day.
CHARACTER: I am here with you. Tell me what happened, one piece at a time.
"""
    result = append_dialogue_examples(dialogue_path, examples_text)
    print('Added pairs:', result['added'])
    print('Total pairs:', result['total_pairs'])
    for warning in result['warnings']:
        print('Warning:', warning)

## Section 6 Validation & Takeaways

Runs a lightweight validation check for the V1.3 notebook state. This confirms that built-in characters are available and that the selected custom dialogue file exists when needed.


In [ ]:
characters = load_characters(CHARACTERS_PATH)
assert len([c for c in characters if not c.get('is_custom')]) >= 6
assert Path(persona.dialogue_file).exists() or not character.get('is_custom')
print('Built-in character count:', len([c for c in characters if not c.get('is_custom')]))
print('Custom character count:', len([c for c in characters if c.get('is_custom')]))
print('V1.3 milestone notebook validation passed.')

### Takeaways

* V1.3 keeps the V1.2 persona chatbot flow and adds guided custom AI creation.
* Character memory is saved per character, with Google Drive persistence when running in Colab.
* Custom dialogue examples can be appended from the notebook for prompt and style testing.
* RAG upload, LoRA/QLoRA training, adapter loading, and export/import packaging remain future milestones.
